In [58]:
import pandas as pd

In [59]:
pops_raw = pd.read_csv("../data/raw/API_SP.POP.TOTL_DS2_en_csv_v2_7/API_SP.POP.TOTL_DS2_en_csv_v2_7.csv", skiprows=4)
gnppc_raw = pd.read_csv("../data/raw/API_NY.GNP.PCAP.CD_DS2_en_csv_v2_230/API_NY.GNP.PCAP.CD_DS2_en_csv_v2_230.csv", skiprows=4)

In [60]:
gnppc_raw.columns

Index(['Country Name', 'Country Code', 'Indicator Name', 'Indicator Code',
       '1960', '1961', '1962', '1963', '1964', '1965', '1966', '1967', '1968',
       '1969', '1970', '1971', '1972', '1973', '1974', '1975', '1976', '1977',
       '1978', '1979', '1980', '1981', '1982', '1983', '1984', '1985', '1986',
       '1987', '1988', '1989', '1990', '1991', '1992', '1993', '1994', '1995',
       '1996', '1997', '1998', '1999', '2000', '2001', '2002', '2003', '2004',
       '2005', '2006', '2007', '2008', '2009', '2010', '2011', '2012', '2013',
       '2014', '2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022',
       '2023', '2024', 'Unnamed: 69'],
      dtype='str')

In [67]:
gnppc

,Country Name,Country Code,Year,GDP per capita
0,Aruba,ABW,1960,NaN
1,Africa Eastern and Southern,AFE,1960,NaN
2,Afghanistan,AFG,1960,NaN
3,Africa Western and Central,AFW,1960,NaN
4,Angola,AGO,1960,NaN
...,...,...,...,...
17285,Kosovo,XKX,2024,6910.0
17286,"Yemen, Rep.",YEM,2024,NaN
17287,South Africa,ZAF,2024,6110.0
17288,Zambia,ZMB,2024,1220.0


In [66]:
pops = (
	pops_raw
	.drop(columns=["Unnamed: 69", "Indicator Name", "Indicator Code"])
	.melt(id_vars=["Country Name", "Country Code"], var_name="Year", value_name="Population")
)
gnppc = (
	gnppc_raw
	.drop(columns=["Unnamed: 69", "Indicator Name", "Indicator Code"])
	.melt(id_vars=["Country Name", "Country Code"], var_name="Year", value_name="GDP per capita")
)
pops_and_gnppc = pops.merge(gnppc, on=["Country Name", "Country Code", "Year"])
pops_and_gnppc = pops_and_gnppc.sort_values(by=["Country Code", "Year"]).groupby(["Country Code"]).agg({
	"Population": lambda x: x.dropna().iloc[-1] if not x.dropna().empty else None,
	"GNP per capita": lambda x: x.dropna().iloc[-1] if not x.dropna().empty else None
}).reset_index()
pops_and_gnppc.to_csv("../data/clean/pops_and_gnppc.csv", index=False)

KeyError: "Label(s) ['GNP per capita'] do not exist"

In [65]:
pops_and_gnppc

,Country Code,Population,GDP per capita
0,ABW,107995.0,35570.000000
1,AFE,769280888.0,1579.103570
2,AFG,42647492.0,370.000000
3,AFW,521764076.0,1624.316384
4,AGO,37885849.0,2840.000000
...,...,...,...
261,XKX,1594353.0,6910.000000
262,YEM,40583164.0,740.000000
263,ZAF,64007187.0,6110.000000
264,ZMB,21314956.0,1220.000000


In [62]:
country_groups = pd.read_excel("../data/raw/CLASS_2025_10_07.xlsx")
ida_ibrd = country_groups[country_groups['Lending category'].notna()]
ida_ibrd['ida'] = ida_ibrd['Lending category'].isin(["IDA", "Blend"])
ida_ibrd['ibrd'] = ida_ibrd['Lending category'].isin(["IBRD", "Blend"])

ida_ibrd = ida_ibrd.set_index('Code')
ida_ibrd.to_csv("../data/clean/ida_ibrd.csv")

In [63]:
pops_and_gnppc_wb = pops_and_gnppc.merge(ida_ibrd, left_on="Country Code", right_index=True)

In [ ]:
# Define lower-middle income countries (GNI per capita between $1,136 and $4,495)
# Using ISO3 codes for FY2026
lmic_countries = {
	'AGO': 'Angola',
	'BGD': 'Bangladesh',
	'BEN': 'Benin',
	'BTN': 'Bhutan',
	'BOL': 'Bolivia',
	'KHM': 'Cambodia',
	'CMR': 'Cameroon',
	'COM': 'Comoros',
	'COG': 'Congo, Rep.',
	'CIV': 'Côte d\'Ivoire',
	'DJI': 'Djibouti',
	'EGY': 'Egypt, Arab Rep.',
	'SWZ': 'Eswatini',
	'GHA': 'Ghana',
	'GIN': 'Guinea',
	'HTI': 'Haiti',
	'HND': 'Honduras',
	'IND': 'India',
	'JOR': 'Jordan',
	'KEN': 'Kenya',
	'KIR': 'Kiribati',
	'KGZ': 'Kyrgyz Republic',
	'LAO': 'Lao PDR',
	'LBN': 'Lebanon',
	'LSO': 'Lesotho',
	'MRT': 'Mauritania',
	'FSM': 'Micronesia, Fed. Sts.',
	'MAR': 'Morocco',
	'MMR': 'Myanmar',
	'NPL': 'Nepal',
	'NIC': 'Nicaragua',
	'NGA': 'Nigeria',
	'PAK': 'Pakistan',
	'PNG': 'Papua New Guinea',
	'PHL': 'Philippines',
	'WSM': 'Samoa',
	'STP': 'São Tomé and Príncipe',
	'SEN': 'Senegal',
	'SLB': 'Solomon Islands',
	'LKA': 'Sri Lanka',
	'TJK': 'Tajikistan',
	'TZA': 'Tanzania',
	'TLS': 'Timor-Leste',
	'TUN': 'Tunisia',
	'UZB': 'Uzbekistan',
	'VUT': 'Vanuatu',
	'VNM': 'Vietnam',
	'ZMB': 'Zambia'
}


In [70]:
# Calculate population sums for different groups
results = {}

# IDA countries
ida_mask = pops_and_gnppc_wb['ida'] == True
results['IDA'] = pops_and_gnppc_wb[ida_mask]['Population'].sum()

# IDA + IBRD countries
ida_ibrd_mask = (pops_and_gnppc_wb['ida'] == True) | (pops_and_gnppc_wb['ibrd'] == True)
results['IDA+IBRD'] = pops_and_gnppc_wb[ida_ibrd_mask]['Population'].sum()

# IDA + IBRD excluding India and China
exclude_india_china = ~pops_and_gnppc_wb['Country Code'].isin(['IND', 'CHN'])
results['IDA+IBRD-India&China'] = pops_and_gnppc_wb[ida_ibrd_mask & exclude_india_china]['Population'].sum()

# IDA + LMIC IBRD countries
lmic_mask = pops_and_gnppc_wb['Country Code'].isin(lmic_countries.keys())
ida_lmic_ibrd_mask = (pops_and_gnppc_wb['ida'] == True) | ((pops_and_gnppc_wb['ibrd'] == True) & lmic_mask)
results['IDA+LMIC IBRD'] = pops_and_gnppc_wb[ida_lmic_ibrd_mask]['Population'].sum()

# Display results
for group, population in results.items():
	print(f"{group}: {population:,.0f}")

# Print list of lower-middle income IBRD countries
print("\nLower-middle income IBRD countries:")
ibrd_only_mask = (pops_and_gnppc_wb['ibrd'] == True) & (pops_and_gnppc_wb['ida'] == False)
lmic_ibrd = pops_and_gnppc_wb[lmic_mask & ibrd_only_mask]
for _, row in lmic_ibrd.iterrows():
	print(f"  {row['Economy']} ({row['Country Code']}): Population={row['Population']:,.0f}, GNP per capita=${row['GDP per capita']:,.0f}")


IDA: 1,946,800,545
IDA+IBRD: 6,926,222,113
IDA+IBRD-India&China: 4,066,311,322
IDA+LMIC IBRD: 3,849,122,234

Lower-middle income IBRD countries:
  Angola (AGO): Population=37,885,849, GNP per capita=$2,840
  Bolivia (BOL): Population=12,413,315, GNP per capita=$4,160
  Egypt, Arab Rep. (EGY): Population=116,538,258, GNP per capita=$3,510
  India (IND): Population=1,450,935,791, GNP per capita=$2,650
  Jordan (JOR): Population=11,552,876, GNP per capita=$4,430
  Lebanon (LBN): Population=5,805,962, GNP per capita=$3,700
  Morocco (MAR): Population=38,081,173, GNP per capita=$3,840
  Philippines (PHL): Population=115,843,670, GNP per capita=$4,470
  Tunisia (TUN): Population=12,277,109, GNP per capita=$3,880
  Vietnam (VNM): Population=100,987,686, GNP per capita=$4,490
